# M5.8 · Diffusion II — koşullu üretim, sınıflandırıcı ve sınıflandırıcısız yönlendirme, latent diffusion

DDPM'den Stable Diffusion'a: ileri sürecin ölçeklemesi (varyansın 1'e yakınsaması), doğrusal ve kosinüs gürültü takvimleri ve SNR; koşullu gürültü tahmini, sınıflandırıcı yönlendirmesi (Dhariwal & Nichol 2021) ve sınıflandırıcısız yönlendirme (Ho & Salimans 2022) — ikisinin kesin modellerde neden aynı olduğu, yönlendirme ölçeğinin kesinlik–çeşitlilik ödünleşimi ve "aşırı doyma"; latent diffusion (Rombach ve ark. 2022): önce otomatik kodlayıcıyla sıkıştır, gizli uzayda difüze et. NumPy'da Gauss karışımı için kapalı formda "kâhin" ε ile yönlendirilmiş DDIM, PyTorch'ta etiket bırakmalı koşullu bir ağ ve CFG, sonda piksel uzayı vs gizli uzay difüzyonu. Ekler: takvimler, DDIM ters çevirme ile düzenleme, ε/x₀/v/skor parametrizasyonları.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-8-diffusion-guidance.html](https://m-gonul.github.io/pytorch-lab/m5-8-diffusion-guidance.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_8_diffusion_guidance.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        close(need(ns, \"cfg\")([0.2, -1.0], [0.5, 1.0], 3), [1.1, 5.0], what=\"CFG\")\n        close(ns[\"cfg\"]([0.2], [0.5], 0), [0.2], what=\"s = 0 koşulsuz\")\n        close(ns[\"cfg\"]([0.2], [0.5], 1), [0.5], what=\"s = 1 koşullu\")\n        close(need(ns, \"classifier_guided\")(0.4, 0.75, 2, 0.3), 0.1, what=\"sınıflandırıcı yönlendirmesi\")\n        close(need(ns, \"guided_gauss\")(0, 1, 2, 1, 3), (1.5, 0.25), what=\"yönlendirilmiş Gauss\")\n        close(ns[\"guided_gauss\"](0, 1, 2, 1, 1), (1.0, 0.5), what=\"s = 1: gerçek posterior\")\n        close(need(ns, \"cosine_abar\")(0, 1000), 1.0, what=\"t = 0\")\n        close(ns[\"cosine_abar\"](500, 1000), math.cos(0.508 / 1.008 * math.pi / 2) ** 2 / math.cos(0.008 / 1.008 * math.pi / 2) ** 2, what=\"kosinüs ortası\")\n        close(need(ns, \"linear_abar\")(2, 1000), (1 - 1e-4) * (1 - 1e-4 - (0.02 - 1e-4) / 999), what=\"doğrusal\")\n        close(need(ns, \"snr\")(0.75), 3.0, what=\"SNR\")\n        rows = []\n        for s in (0, 0.5, 1, 2, 4, 10):\n            m, v = ns[\"guided_gauss\"](0, 1, 2, 1, s)\n            rows.append([s, f\"{m:.3f}\", f\"{math.sqrt(v):.3f}\"])\n        R_.table(\"Önsel N(0, 1), olabilirlik N(x; 2, 1): p(x)·p(y|x)^s\", [\"s\", \"ortalama\", \"std\"], rows,\n                 note=\"Yönlendirme ölçeği olabilirliği 'keskinleştirir': s = 1 gerçek koşullu dağılım, s > 1 dağılımı olabilirliğin tepesine doğru iter ve daraltır (kesinlik–çeşitlilik ödünleşimi), s < 1 önsele doğru yumuşatır.\")\n    R_.test(\"formüller\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: (max_loops(src, 0, func=\"mixture_eps\"), max_loops(src, 0, func=\"class_logprob_grad\"), max_loops(src, 1, func=\"ddim\")))\n    MU, LAB, SIG = ns[\"MU\"], ns[\"LAB\"], ns[\"SIG\"]\n    wu = np.full(8, 1 / 8)\n\n    def logp(x, abar, w):\n        v = abar * SIG ** 2 + 1 - abar\n        d2 = ((x[:, None] - np.sqrt(abar) * MU[None]) ** 2).sum(-1)\n        return np.log((w[None] * np.exp(-d2 / (2 * v))).sum(1))\n\n    def t_exact():\n        rng = np.random.default_rng(0)\n        x = rng.normal(0, 2, (5, 2))\n        e = need(ns, \"mixture_eps\")(x, 0.6, wu)\n        h = 1e-6\n        g = np.stack([(logp(x + h * d, 0.6, wu) - logp(x, 0.6, wu)) / h for d in np.eye(2)], 1)\n        close(e, -np.sqrt(0.4) * g, atol=1e-4, what=\"ε = −σ·∇log p(x_t)\")\n        wc = (LAB == 1) / 2.0\n        lp_y = lambda z: logp(z, 0.6, wc) + np.log(0.25) - logp(z, 0.6, wu)\n        gy = np.stack([(lp_y(x + h * d) - lp_y(x)) / h for d in np.eye(2)], 1)\n        close(need(ns, \"class_logprob_grad\")(x, 0.6, 1), gy, atol=1e-4, what=\"∇log p(y|x_t) (Bayes)\")\n        for s in (0.5, 3.0):\n            cg = ns[\"mixture_eps\"](x, 0.6, wu) - s * np.sqrt(0.4) * ns[\"class_logprob_grad\"](x, 0.6, 1)\n            cf = ns[\"mixture_eps\"](x, 0.6, wu) + s * (ns[\"mixture_eps\"](x, 0.6, wc) - ns[\"mixture_eps\"](x, 0.6, wu))\n            close(cg, cf, atol=1e-10, what=f\"sınıflandırıcı yönlendirmesi = CFG (s = {s})\")\n    R_.test(\"kesin skorlar\", t_exact)\n\n    def t_sample():\n        betas = np.linspace(1e-4, 0.02, 1000)\n        ab = np.cumprod(1 - betas)[::5]\n        xT = np.random.default_rng(1).normal(size=(3000, 2))\n        x0 = need(ns, \"ddim\")(xT, ab, lambda x, a: ns[\"mixture_eps\"](x, a, wu))\n        near = np.argmin(((x0[:, None] - MU[None]) ** 2).sum(-1), 1)\n        cnt = np.bincount(near, minlength=8) / len(x0)\n        ensure(np.all(np.abs(cnt - 1 / 8) < 0.03), \"koşulsuz: 8 mod eşit\")\n        wc = (LAB == 1) / 2.0\n        rows, R = [], {}\n        for s in (0.0, 1.0, 2.0, 4.0, 8.0):\n            f = lambda x, a: ns[\"mixture_eps\"](x, a, wu) + s * (ns[\"mixture_eps\"](x, a, wc) - ns[\"mixture_eps\"](x, a, wu))\n            x0 = ns[\"ddim\"](xT, ab, f)\n            d2 = ((x0[:, None] - MU[None]) ** 2).sum(-1)\n            r = np.exp(-(d2 - d2.min(1, keepdims=True)) / (2 * SIG ** 2))\n            r /= r.sum(1, keepdims=True)\n            py = r[:, LAB == 1].sum(1).mean()\n            k = np.argmin(d2, 1)\n            two = np.bincount(k[LAB[k] == 1], minlength=8)[LAB == 1]\n            R[s] = (py, np.linalg.norm(x0, axis=1).mean(), np.sqrt(d2.min(1)).mean())\n            rows.append([s, f\"{py:.3f}\", f\"{two[0]}/{two[1]}\", f\"{R[s][1]:.2f}\", f\"{R[s][2]:.2f}\"])\n        ensure(R[0.0][0] < 0.35 and R[1.0][0] > 0.98, \"s = 0 koşulsuz, s = 1 koşullu\")\n        ensure(R[8.0][1] > R[1.0][1] + 0.5, \"büyük s: örnekler dışa taşar\")\n        R_.table(\"Kesin ε ile yönlendirilmiş DDIM (200 adım), hedef sınıf 1 (açı 45° ve 225°)\", [\"s\", \"ort. p(y = 1 | x₀)\", \"iki modun payı\", \"ort. yarıçap (veri: 4)\", \"en yakın merkeze uzaklık (veri ≈ 0.63)\"], rows,\n                 note=\"Model kusursuz olduğu hâlde s > 1 dağılımı değiştirir: p(x)·p(y|x)^s. Sınıf zaten kesin olduğundan kesinlik doymuştur; bunun yerine örnekler diğer sınıflardan 'kaçmak' için dışarı itilir (yarıçap 4'ün üstüne) — görüntülerdeki aşırı doymuş renkler ve abartılı özelliklerin oyuncak karşılığı. Her iki mod korunur: yönlendirme sınıf içi modları değil, sınıflar arası ayrımı keskinleştirir.\")\n    R_.test(\"yönlendirilmiş örnekleme\", t_sample)\n",
 "d": "import time\nimport torch\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        MU, LAB = ns[\"MU\"], ns[\"LAB\"]\n        torch.manual_seed(0)\n        m = need(ns, \"EpsNet\")(4, 128)\n        ensure(tuple(m(torch.randn(3, 2), torch.tensor([0, 5, 199]), torch.tensor([0, 4, 2])).shape) == (3, 2), \"çıkış (B, 2)\")\n        t0 = time.perf_counter()\n        need(ns, \"train\")(m, 4000, 0.15, 0)\n        tt = time.perf_counter() - t0\n        rows, R = [], {}\n        for s in (0.0, 1.0, 3.0, 6.0):\n            x = need(ns, \"sample\")(m, 2000, 1, s, 1)\n            d2 = torch.cdist(x, MU) ** 2\n            r = torch.softmax(-d2 / 0.5, 1)\n            py = r[:, LAB == 1].sum(1).mean().item()\n            R[s] = (py, x.norm(dim=1).mean().item(), d2.min(1).values.sqrt().median().item())\n            rows.append([s, f\"{py:.3f}\", f\"{R[s][1]:.2f}\", f\"{R[s][2]:.2f}\"])\n        ensure(R[0.0][0] < 0.5 and R[1.0][0] > 0.9, f\"koşulsuz/koşullu {R}\")\n        ensure(R[6.0][1] > R[1.0][1] + 0.5 and R[6.0][2] > R[1.0][2], \"büyük s: dışa taşma\")\n        R_.table(f\"Etiket bırakmalı (%15) koşullu ağ, 4000 adım ({tt:.0f} sn); hedef sınıf 1\", [\"s\", \"ort. p(y = 1 | x)\", \"ort. yarıçap (veri 4)\", \"medyan en yakın merkez uzaklığı\"], rows,\n                 note=\"Tek ağ, boş token'la koşulsuz, etiketle koşullu ε verir. s = 0 sınıfı yok sayar (≈ 0.25), s = 1 koşullu dağılımdan örnekler; s arttıkça c aşamasındaki kesin modeldeki gibi örnekler sınıf ayrımı yönünde dışa itilir. Öğrenilmiş modelde koşullu ε'nun kusurları yönlendirmeyle büyütülür — pratikte s ≈ 3–8 (görüntü) bu yüzden ayarlanır.\")\n    R_.test(\"CFG ile koşullu difüzyon\", t)\n",
 "e": "import time\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.datasets import load_digits\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        X, y = load_digits(return_X_y=True)\n        X = torch.tensor(X, dtype=torch.float32) / 16\n        y = torch.tensor(y)\n        torch.manual_seed(0)\n        enc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 8))\n        dec = nn.Sequential(nn.Linear(8, 128), nn.ReLU(), nn.Linear(128, 64), nn.Sigmoid())\n        o = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), 3e-3)\n        for _ in range(150):\n            p = torch.randperm(len(X))\n            for i in range(0, len(X), 64):\n                b = X[p[i:i + 64]]\n                loss = F.mse_loss(dec(enc(b)), b)\n                o.zero_grad()\n                loss.backward()\n                o.step()\n        clf = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))\n        o = torch.optim.Adam(clf.parameters(), 1e-3)\n        for _ in range(100):\n            p = torch.randperm(len(X))\n            for i in range(0, len(X), 64):\n                b = p[i:i + 64]\n                loss = F.cross_entropy(clf(X[b]), y[b])\n                o.zero_grad()\n                loss.backward()\n                o.step()\n        with torch.no_grad():\n            Z = enc(X)\n            mu, sd = Z.mean(0), Z.std(0)\n            Z = (Z - mu) / sd\n        labels = torch.arange(10).repeat(50)\n        rows, R = [], {}\n        for name, data, D in ((\"piksel uzayı (64)\", X * 2 - 1, 64), (\"gizli uzay (8) + kod çözücü\", Z, 8)):\n            t0 = time.perf_counter()\n            m = need(ns, \"train_diffusion\")(data, y, 3000, 0)\n            tt = time.perf_counter() - t0\n            for s in (1.0, 3.0):\n                t0 = time.perf_counter()\n                z = need(ns, \"sample\")(m, labels, D, s, 1)\n                ts = time.perf_counter() - t0\n                with torch.no_grad():\n                    img = ((z + 1) / 2).clamp(0, 1) if D == 64 else dec(z * sd + mu)\n                    pr = clf(img).softmax(1)\n                acc = (pr.argmax(1) == labels).float().mean().item()\n                R[(D, s)] = acc\n                rows.append([name, s, f\"{tt:.1f} sn\", f\"{ts:.2f} sn\", f\"{acc:.3f}\", f\"{pr.max(1).values.mean().item():.3f}\"])\n        ensure(R[(8, 3.0)] > 0.85 and R[(8, 3.0)] > R[(64, 3.0)] + 0.2, f\"gizli uzay aynı bütçede çok daha iyi {R}\")\n        ensure(R[(8, 3.0)] >= R[(8, 1.0)] - 0.02, \"CFG sınıf doğruluğunu korur/artırır\")\n        R_.table(\"Koşullu difüzyon, rakamlar, 3000 adım (aynı MLP bütçesi), 500 örnek\", [\"uzay\", \"s\", \"eğitim\", \"örnekleme\", \"sınıf doğruluğu\", \"ort. güven\"], rows,\n                 note=\"Gizli uzay 8 kat küçük ve otomatik kodlayıcı piksel düzeyindeki ayrıntıyı (kalem kalınlığı, gürültü) zaten üstlenmiş: difüzyon modeli yalnız 'hangi rakam, hangi stil' yapısını öğrenir. Aynı eğitim bütçesinde piksel uzayındaki model çok geride kalır. Gerçek latent diffusion'da U-Net ve 8× küçültme ile hesap kazancı da büyüktür (burada MLP gizli boyutu baskın olduğundan süreler yakın).\")\n    R_.test(\"piksel vs gizli uzay\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"linear_abar\", \"cosine_abar\", \"log_snr\", \"wasted_fraction\")])\n\n    def t():\n        L = need(ns, \"linear_abar\")(1000)\n        C = need(ns, \"cosine_abar\")(1000)\n        close(L[1], (1 - 1e-4) * (1 - 1e-4 - (0.02 - 1e-4) / 999), what=\"doğrusal\")\n        close(C[499], np.cos(0.508 / 1.008 * np.pi / 2) ** 2 / np.cos(0.008 / 1.008 * np.pi / 2) ** 2, atol=1e-6, what=\"kosinüs ortası\")\n        ensure(np.all(np.diff(C) <= 0) and C[-1] < 1e-3, \"monoton, sonda ~0\")\n        wl, wc = need(ns, \"wasted_fraction\")(L), ns[\"wasted_fraction\"](C)\n        ensure(wl > 0.2 and wc < 0.1, f\"doğrusal takvim sonda israf ({wl:.2f} vs {wc:.2f})\")\n        rows = []\n        for q in (0.1, 0.25, 0.5, 0.75, 0.9):\n            i = int(q * 1000) - 1\n            rows.append([q, f\"{L[i]:.4f}\", f\"{C[i]:.4f}\", f\"{need(ns, 'log_snr')(L[i:i + 1])[0]:+.2f}\", f\"{ns['log_snr'](C[i:i + 1])[0]:+.2f}\"])\n        R_.table(\"T = 1000: ᾱ ve log SNR\", [\"t/T\", \"ᾱ doğrusal\", \"ᾱ kosinüs\", \"log SNR doğrusal\", \"log SNR kosinüs\"], rows,\n                 note=f\"Doğrusal takvim sinyali çok hızlı yok eder: adımların %{wl * 100:.0f}'ında ᾱ < 0.01 (neredeyse saf gürültü — model bu adımlarda az şey öğrenir). Kosinüs takvimi log SNR'yi neredeyse doğrusal düşürür: bilgi yok oluşu zamana eşit yayılır (Nichol & Dhariwal 2021, Şekil 3). Düşük çözünürlüklü görüntülerde bu fark özellikle önemlidir.\")\n    R_.test(\"takvimler\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        MU, LAB = ns[\"MU\"], ns[\"LAB\"]\n        wu = np.full(8, 1 / 8)\n        rng = np.random.default_rng(0)\n        k = rng.integers(0, 8, 500)\n        x0 = MU[k] + 0.5 * rng.normal(size=(500, 2))\n        rows = []\n        errs = {}\n        for steps in (10, 50, 200):\n            ab = np.cumprod(1 - np.linspace(1e-4, 0.02, 1000))[::1000 // steps]\n            f = lambda x, a: ns[\"mixture_eps\"](x, a, wu)\n            xT = need(ns, \"ddim_invert\")(x0, ab, f)\n            rec = need(ns, \"ddim\")(xT, ab, f)\n            errs[steps] = np.abs(rec - x0).mean()\n            rows.append([steps, f\"{errs[steps]:.4f}\", f\"{xT.std():.2f}\"])\n        ensure(errs[200] < 0.05 and errs[200] < errs[10], f\"ters çevirme hatası {errs}\")\n        ab = np.cumprod(1 - np.linspace(1e-4, 0.02, 1000))[::5]\n        f = lambda x, a: ns[\"mixture_eps\"](x, a, wu)\n        sel = LAB[k] == 0\n        xT = ns[\"ddim_invert\"](x0[sel], ab, f)\n        wc = (LAB == 2) / 2.0\n        edit = ns[\"ddim\"](xT, ab, lambda x, a: ns[\"mixture_eps\"](x, a, wu) + 4 * (ns[\"mixture_eps\"](x, a, wc) - ns[\"mixture_eps\"](x, a, wu)))\n        kk = np.argmin(((edit[:, None] - MU[None]) ** 2).sum(-1), 1)\n        frac = (LAB[kk] == 2).mean()\n        ensure(frac > 0.9, f\"düzenleme sınıf 2'ye taşır ({frac:.2f})\")\n        R_.table(\"DDIM ters çevirme: x₀ → x_T → x₀ (kesin ε)\", [\"adım\", \"ort. |x̂₀ − x₀|\", \"x_T std\"], rows,\n                 note=f\"Deterministik DDIM bir ODE'nin ayrıklaştırmasıdır: tersine çalıştırınca görüntüyü gürültü uzayında bir 'koda' çevirir, adım sayısı arttıkça yeniden kurulum kusursuzlaşır. Sınıf 0'dan ters çevrilen örnekler, aynı x_T'den sınıf 2 yönlendirmesiyle (s = 4) çözülünce %{frac * 100:.0f}'i sınıf 2'ye geçer — gürültü kodu konum/stil bilgisini taşırken koşul anlamı değiştirir (görüntü düzenlemenin temeli: SDEdit, prompt-to-prompt, null-text inversion).\")\n    R_.test(\"ters çevirme ve düzenleme\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        x0, eps = rng.normal(size=(5, 3)), rng.normal(size=(5, 3))\n        rows = []\n        for abar in (0.999, 0.5, 0.01):\n            a, s = np.sqrt(abar), np.sqrt(1 - abar)\n            xt = a * x0 + s * eps\n            close(need(ns, \"eps_to_x0\")(xt, eps, abar), x0, atol=1e-9, what=\"ε → x₀\")\n            close(need(ns, \"x0_to_eps\")(xt, x0, abar), eps, atol=1e-9, what=\"x₀ → ε\")\n            v = a * eps - s * x0\n            close(need(ns, \"eps_to_v\")(xt, eps, abar), v, atol=1e-9, what=\"ε → v\")\n            close(need(ns, \"v_to_eps\")(xt, v, abar), eps, atol=1e-9, what=\"v → ε\")\n            close(need(ns, \"eps_to_score\")(eps, abar), -eps / s, atol=1e-9, what=\"skor\")\n            err = 0.01 * rng.normal(size=(5, 3))\n            rows.append([abar, f\"{np.abs(ns['eps_to_x0'](xt, eps + err, abar) - x0).mean():.4f}\", f\"{np.abs((a * xt - s * (v + err)) - x0).mean():.4f}\"])\n        R_.table(\"ε'da ya da v'de 0.01'lik tahmin hatasının x̂₀'a etkisi\", [\"ᾱ\", \"ε-tahmini → |Δx̂₀|\", \"v-tahmini → |Δx̂₀|\"], rows,\n                 note=\"x̂₀ = (x_t − σε)/α: ᾱ → 0'da (saf gürültü) ε'daki küçük bir hata 1/α kadar büyütülür. v = αε − σx₀ ile x̂₀ = αx_t − σv ve hata büyütmesi en fazla 1'dir: v-parametrizasyonu yüksek gürültü ucunda kararlıdır (progresif damıtma, Stable Diffusion 2). Hepsi aynı bilgiyi taşır; skor = −ε/σ.\")\n    R_.test(\"parametrizasyonlar\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_8_diffusion_guidance as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**DDPM özeti** (Ho ve ark. 2020): $x_0$'a her adımda varyansı $\beta_t$ olan izotropik Gauss gürültüsü eklenir ve değerler $\sqrt{1-\beta_t}$ ile ölçeklenir; böylece varyans 1'e yakınsar ($(1-\beta_t)+\beta_t=1$). Kısa yol: $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$. Ağ $\varepsilon$'u tahmin eder; geri süreç $p_\theta(x_{t-1}|x_t)$ Markov'dur. Özgün makalede T = 1000 ve doğrusal takvim; Nichol & Dhariwal (2021) T = 4000 ve başta/sonda daha yavaş değişen **kosinüs takvimi** önerdi ve DDPM'leri GAN'ların önüne geçiren iyileştirmeler yaptı.

**Koşullu üretim ve yönlendirme:** $\nabla\log p(x_t|y)=\nabla\log p(x_t)+\nabla\log p(y|x_t)$. **Sınıflandırıcı yönlendirmesi** (Dhariwal & Nichol 2021): gürültülü görüntüler üzerinde ayrı bir sınıflandırıcı eğitilir; ε tahmini onun gradyanıyla kaydırılır: $\hat\varepsilon=\varepsilon_\theta-s\sqrt{1-\bar\alpha_t}\nabla_{x_t}\log p_\phi(y|x_t)$ — ölçek s > 1, $p(x)p(y|x)^s$'den örneklemeye karşılık gelir: daha keskin, daha az çeşitli. **Sınıflandırıcısız yönlendirme** (Ho & Salimans 2022): ayrı sınıflandırıcı yok; tek ağ eğitimde etiketi bir olasılıkla boş token'la değiştirerek hem koşullu hem koşulsuz modeli öğrenir; örneklemede $\hat\varepsilon=\varepsilon_u+s(\varepsilon_c-\varepsilon_u)$. Kesin modellerde ikisi özdeştir, çünkü $\varepsilon_c-\varepsilon_u=-\sqrt{1-\bar\alpha}\,\nabla\log p(y|x_t)$ (Bayes).

**Latent diffusion** (Rombach ve ark. 2021/22): güçlü bir otomatik kodlayıcı her görüntüyü çok daha küçük bir gizli uzaya sıkıştırır, difüzyon orada yapılır, sonuç kod çözücüyle görüntüye açılır: üretim ve eğitim maliyeti büyük ölçüde düşer, kalite yüksek. Metin, görüntü vb. koşullar U-Net'e cross-attention ile verilir. **Stable Diffusion** (Ağustos 2022) bu modelin açık kaynak sürümüdür.

## 1 · Elle

CFG: $0.2+3(0.5-0.2)=1.1$ — s = 1 koşullu modeldir, s = 0 koşulsuz; s > 1 koşullu yönde dışa doğru ekstrapolasyon.

Sınıflandırıcı yönlendirmesi: $0.4-2\cdot\sqrt{0.25}\cdot0.3=0.1$.

Gauss örneği: $p(x)p(y|x)^s\propto e^{-x^2/2}e^{-s(x-2)^2/2}$ → hassasiyet $1+s=4$, ortalama $2s/(1+s)=1.5$, varyans 0.25. s = 1 gerçek posterior (ortalama 1, varyans 0.5); s büyüdükçe ortalama olabilirlik tepesine (2) gider, varyans 0'a: kesinlik ↑, çeşitlilik ↓.

Kosinüs: $f(t)=\cos^2\!\big(\frac{t/T+0.008}{1.008}\frac\pi2\big)$, $\bar\alpha=f(t)/f(0)$; ortada ≈ 0.494 — sinyal ve gürültü tam ortada dengelenir (doğrusal takvimde ortada $\bar\alpha\approx0.08$). SNR = 0.75/0.25 = 3. Latent: 786 432 / 16 384 = 48.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yönlendirme**

- `cfg`: CFG: $\varepsilon_u=0.2$, $\varepsilon_c=0.5$, s = 3 → $\hat\varepsilon$
- `cg`: Sınıflandırıcı yönlendirmesi: ε = 0.4, $\bar\alpha=0.75$, s = 2, $\nabla\log p(y|x_t)=0.3$
- `gmean`: $p(x)=N(0,1)$, $p(y|x)\propto N(x;2,1)$, $p(x)p(y|x)^s$ ile s = 3: ortalama
- `gvar`: Aynı dağılımın varyansı

**Takvimler ve latent diffusion**

- `cosab`: Kosinüs takvimi (s = 0.008): $\bar\alpha$ tam ortada (t = T/2)
- `snr`: $\bar\alpha=0.75$'te SNR = $\bar\alpha/(1-\bar\alpha)$
- `lat`: 512 × 512 × 3 görüntü → 64 × 64 × 4 gizli kod: boyut küçülme oranı

In [ ]:
cevaplar = dict(
    cfg=None,
    cg=None,
    gmean=None,
    gvar=None,
    cosab=None,
    snr=None,
    lat=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"cfg":[1.1,1e-9],"cg":[0.1,1e-9],"gmean":[1.5,1e-9],"gvar":[0.25,1e-9],"cosab":[0.49384359044063775,0.000001],"snr":[3,1e-9],"lat":[48,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `cfg(eps_u, eps_c, s)` (listeler).
- `classifier_guided(eps, abar, s, grad)`.
- `guided_gauss(m0, v0, ml, vl, s)` — $N(m_0,v_0)\cdot N(x;m_l,v_l)^s$'nin ortalaması ve varyansı.
- `cosine_abar(t, T, s=0.008)` ve `linear_abar(t, T, b1=1e-4, bT=0.02)` ($\prod_{i\le t}(1-\beta_i)$, t = 1..T).
- `snr(abar)`.

In [ ]:
%%writefile stage_b.py
import math


def cfg(eps_u, eps_c, s):
    ...


def classifier_guided(eps, abar, s, grad):
    ...


def guided_gauss(m0, v0, ml, vl, s):
    ...


def cosine_abar(t, T, s=0.008):
    ...


def linear_abar(t, T, b1=1e-4, bT=0.02):
    ...


def snr(abar):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy. Veri: yarıçapı 4 olan halkada 8 Gauss (σ = 0.5); bileşen k'nin sınıfı k mod 4 (her sınıfın karşılıklı iki modu var). Gauss karışımı gürültülendiğinde de karışım kalır: $x_t\sim\sum\pi_kN(\sqrt{\bar\alpha}\mu_k,(\bar\alpha\sigma^2+1-\bar\alpha)I)$ → skor kapalı formda.

- `mixture_eps(x, abar, w)` — `x (N, 2)`, bileşen ağırlıkları `w (8,)`: kesin $\varepsilon(x_t)=-\sqrt{1-\bar\alpha}\,\nabla\log p(x_t)$ (sorumluluklarla). Döngüsüz.
- `class_logprob_grad(x, abar, y)` — $\nabla_{x}\log p(y|x_t)$, kesin (döngüsüz).
- `ddim(xT, abars, eps_fn)` — deterministik DDIM, `abars` artan t sırasında; son adımda $\hat x_0$ döndür (tek döngü).

In [ ]:
%%writefile stage_c.py
import numpy as np

ANG = np.arange(8) * np.pi / 4
MU = 4 * np.stack([np.cos(ANG), np.sin(ANG)], 1)
LAB = np.arange(8) % 4
SIG = 0.5


def mixture_eps(x, abar, w):
    ...


def class_logprob_grad(x, abar, y):
    ...


def ddim(xT, abars, eps_fn):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Aynı 8 Gauss halkası, 4 sınıf; T = 200, $\beta$ 10⁻⁴'ten 0.1'e doğrusal.

- `EpsNet(C=4, h=128)` — girdi izdüşümü + sinüzoidal zaman embedding'i + sınıf embedding'i (C + 1 giriş: son indeks boş token), toplanıp SiLU'lu MLP'ye.
- `train(model, steps, p_uncond=0.15, seed=0)` — her batch'te etiketlerin `p_uncond`'u boş token'a çevrilir.
- `sample(model, n, y, s, seed)` — CFG'li deterministik DDIM (her adımda iki değerlendirme).

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

ANG = torch.arange(8) * math.pi / 4
MU = 4 * torch.stack([torch.cos(ANG), torch.sin(ANG)], 1)
LAB = torch.arange(8) % 4
T = 200
BETAS = torch.linspace(1e-4, 0.1, T)
AB = torch.cumprod(1 - BETAS, 0)


def data(n, g):
    k = torch.randint(0, 8, (n,), generator=g)
    return MU[k] + 0.5 * torch.randn(n, 2, generator=g), LAB[k]


class EpsNet(nn.Module):
    def __init__(self, C=4, h=128):
        super().__init__()
        ...

    def forward(self, x, t, y):
        ...


def train(model, steps, p_uncond=0.15, seed=0):
    ...


def sample(model, n, y, s, seed):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Rakamlar (64 piksel). Verilenler: küçük bir otomatik kodlayıcı (8 boyutlu gizli kod) ve rakam sınıflandırıcısı.

- `CondDenoiser(D, C=10, h=256)` — `EpsNet`'in D boyutlu, 10 sınıflı sürümü.
- `train_diffusion(Z, y, steps, seed)` ve `sample(model, labels, D, s, seed)`.
- Karşılaştırma: piksel uzayı (D = 64, [−1, 1]) vs gizli uzay (D = 8, standartlaştırılmış kodlar, sonra kod çözücü) — aynı adım bütçesi, sınıf doğruluğu (sınıflandırıcıyla), güven, süre.

In [ ]:
%%writefile stage_e.py
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 200
BETAS = torch.linspace(1e-4, 0.1, T)
AB = torch.cumprod(1 - BETAS, 0)


class CondDenoiser(nn.Module):
    def __init__(self, D, C=10, h=256):
        super().__init__()
        ...

    def forward(self, x, t, y):
        ...


def train_diffusion(Z, y, steps, seed):
    ...


def sample(model, labels, D, s, seed):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Gürültü takvimleri: doğrusal vs kosinüs, SNR

NumPy.

- `linear_abar(T, b1, bT)` ve `cosine_abar(T, s=0.008)` → `(T,)` diziler (t = 1..T), döngüsüz; kosinüs $\beta_t\le0.999$ kırpmasıyla.
- `log_snr(abar)`.
- `wasted_fraction(abar, thr=0.01)` — $\bar\alpha<$ eşik olan (sinyalin neredeyse tamamen yok olduğu) adımların oranı.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def linear_abar(T, b1=1e-4, bT=0.02):
    ...


def cosine_abar(T, s=0.008):
    ...


def log_snr(abar):
    ...


def wasted_fraction(abar, thr=0.01):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · DDIM ters çevirme ve koşulu değiştirerek düzenleme

NumPy, c'deki kapalı form ε ile.

- `ddim_invert(x0, abars, eps_fn)` — deterministik DDIM'i ters yönde ($x_{t}$'den $x_{t+1}$'e, ε'yu $x_t$'de değerlendirerek) → $x_T$.
- `ddim(xT, abars, eps_fn)` (c'deki gibi).

In [ ]:
%%writefile extra_x2.py
import numpy as np

ANG = np.arange(8) * np.pi / 4
MU = 4 * np.stack([np.cos(ANG), np.sin(ANG)], 1)
LAB = np.arange(8) % 4
SIG = 0.5


def mixture_eps(x, abar, w):
    v = abar * SIG ** 2 + 1 - abar
    m = np.sqrt(abar) * MU
    lw = np.log(np.clip(w, 1e-300, None))[None] - ((x[:, None] - m[None]) ** 2).sum(-1) / (2 * v)
    r = np.exp(lw - lw.max(1, keepdims=True))
    r /= r.sum(1, keepdims=True)
    return np.sqrt(1 - abar) * (x - r @ m) / v


def ddim(xT, abars, eps_fn):
    ...


def ddim_invert(x0, abars, eps_fn):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Parametrizasyonlar: ε, x₀, v ve skor

NumPy. $x_t=\alpha x_0+\sigma\varepsilon$, $\alpha=\sqrt{\bar\alpha}$, $\sigma=\sqrt{1-\bar\alpha}$, $v=\alpha\varepsilon-\sigma x_0$.

- `eps_to_x0(xt, eps, abar)`, `x0_to_eps(xt, x0, abar)`, `eps_to_v(xt, eps, abar)`, `v_to_eps(xt, v, abar)`, `eps_to_score(eps, abar)` ($-\varepsilon/\sigma$).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def eps_to_x0(xt, eps, abar):
    ...


def x0_to_eps(xt, x0, abar):
    ...


def eps_to_v(xt, eps, abar):
    ...


def v_to_eps(xt, v, abar):
    ...


def eps_to_score(eps, abar):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Ho & Salimans (2022): CFG'yi rakamlarda/CIFAR'da

[Ho & Salimans — Classifier-Free Diffusion Guidance (NeurIPS 2021 Workshop / 2022)](https://arxiv.org/abs/2207.12598)

M5.3'teki U-Net'i sınıf embedding'iyle koşullu yap, etiket bırakma olasılığı p_uncond ∈ {0.1, 0.2, 0.5} ile MNIST/CIFAR-10'da eğit. Makale Tablo 1–2'nin küçük sürümü: yönlendirme ağırlığı w ∈ {0, 0.3, 1, 2, 4} için FID ve IS (M5.7'deki ölçücüler); FID–IS eğrisini çiz (Şekil 4). Büyük w'de renk/kontrast aşırı doymasını gözle ve "dinamik eşikleme" (Saharia ve ark. 2022, Imagen) ile düzelt.


**Adımlar**

1. Dinamik eşikleme: x̂₀'ın mutlak değerlerinin p-yüzdeliği s ile kırp ve s'ye böl.

**Kabul ölçütleri**

- w arttıkça IS artar, FID önce düşer sonra artar.
- Dinamik eşikleme büyük w'de FID'i iyileştirir.

## Makale ödevi: Rombach ve ark. (2022): küçük bir latent diffusion

[Rombach, Blattmann, Lorenz, Esser, Ommer — High-Resolution Image Synthesis with Latent Diffusion Models (CVPR 2022)](https://arxiv.org/abs/2112.10752)

CelebA-64 ya da CIFAR-10'da KL düzenlemeli bir konvolüsyonel otomatik kodlayıcı (f ∈ {1, 2, 4, 8} küçültme, M5.6) eğit; her f için gizli uzayda aynı U-Net bütçesiyle difüzyon modeli eğitip FID ve eğitim süresini karşılaştır (makale Şekil 6: f = 4–16 tatlı nokta). Sınıf koşulunu U-Net'e cross-attention ile ver (M4.14), CFG ile örnekle.


**Adımlar**

1. Gizli kodları birim varyansa ölçekle (makalenin 'scale factor'ü).

**Kabul ölçütleri**

- f = 1 (piksel uzayı) en yavaş; f çok büyükse yeniden yapılandırma kalitesi FID'i sınırlar.
- Cross-attention koşullama + CFG sınıf doğruluğunu artırır.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 17 (DDPM)](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Ho, Jain, Abbeel — Denoising Diffusion Probabilistic Models (NeurIPS 2020)](https://arxiv.org/abs/2006.11239) — arXiv
- [Nichol & Dhariwal — Improved Denoising Diffusion Probabilistic Models (ICML 2021)](https://arxiv.org/abs/2102.09672) — arXiv
- [Dhariwal & Nichol — Diffusion Models Beat GANs on Image Synthesis (NeurIPS 2021)](https://arxiv.org/abs/2105.05233) — arXiv
- [Ho & Salimans — Classifier-Free Diffusion Guidance (2022)](https://arxiv.org/abs/2207.12598) — arXiv
- [Rombach ve ark. — High-Resolution Image Synthesis with Latent Diffusion Models (CVPR 2022)](https://arxiv.org/abs/2112.10752) — arXiv
- [Song, Meng, Ermon — Denoising Diffusion Implicit Models (ICLR 2021)](https://arxiv.org/abs/2010.02502) — arXiv
- [Salimans & Ho — Progressive Distillation (v-parametrizasyonu, ICLR 2022)](https://arxiv.org/abs/2202.00512) — arXiv
- [fast.ai Part 2 — Stable Diffusion from the Foundations](https://course.fast.ai/Lessons/part2.html) — Apache-2.0